# 🤖 AI Interface Orchestrator

Provider-agnostic, declarative interface contracts for AI workloads in fh-saas.

This notebook defines:
- Core config and job state contracts
- Protocol seams for backend/store/publisher adapters
- Small declarative `Interface` API for route orchestration
- Polling helper contract for Phase 1 UX

In [ ]:
#| default_exp utils_ai_interface

In [ ]:
#| export
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, runtime_checkable, Callable
import os
import logging

logger = logging.getLogger(__name__)

---
## Core Contracts

In [ ]:
#| export
class JobState(str, Enum):
    queued = 'queued'
    processing = 'processing'
    streaming = 'streaming'
    done = 'done'
    failed = 'failed'
    canceled = 'canceled'


@dataclass
class AIInterfaceConfig:
    max_concurrency: int = 1
    default_poll_interval_ms: int = 1200
    status_endpoint_template: str = '/api/jobs/{job_id}'
    result_endpoint_template: str = '/api/jobs/{job_id}/result'
    max_upload_mb: int = 100
    request_timeout_seconds: int = 900
    storage_mode: str = 'db_blob'

    @classmethod
    def from_env(cls) -> 'AIInterfaceConfig':
        return cls(
            max_concurrency=int(os.getenv('AI_MAX_CONCURRENCY', '1')),
            default_poll_interval_ms=int(os.getenv('AI_POLL_INTERVAL_MS', '1200')),
            status_endpoint_template=os.getenv('AI_STATUS_ENDPOINT_TEMPLATE', '/api/jobs/{job_id}'),
            result_endpoint_template=os.getenv('AI_RESULT_ENDPOINT_TEMPLATE', '/api/jobs/{job_id}/result'),
            max_upload_mb=int(os.getenv('AI_MAX_UPLOAD_MB', '100')),
            request_timeout_seconds=int(os.getenv('AI_REQUEST_TIMEOUT_SECONDS', '900')),
            storage_mode=os.getenv('AI_STORAGE_MODE', 'db_blob'),
        )


@dataclass
class JobRecord:
    id: str
    tenant_id: str
    job_type: str
    state: JobState
    payload: Dict[str, Any] = field(default_factory=dict)
    result: Optional[Dict[str, Any]] = None
    error: Optional[str] = None
    progress: int = 0


@dataclass
class ArtifactRef:
    id: str
    tenant_id: str
    content_type: str
    size_bytes: int
    filename: Optional[str] = None

In [ ]:
#| export
@runtime_checkable
class InferenceBackend(Protocol):
    def run(self, *, fn: Callable[..., Any], payload: Dict[str, Any]) -> Any: ...


@runtime_checkable
class JobStore(Protocol):
    def enqueue(self, tenant_id: str, job_type: str, payload: Dict[str, Any]) -> str: ...
    def get(self, job_id: str): ...
    def cancel(self, job_id: str) -> bool: ...


@runtime_checkable
class ArtifactStore(Protocol):
    def put(self, *, tenant_id: str, content_type: str, content: bytes, filename: Optional[str] = None) -> ArtifactRef: ...
    def get(self, artifact_id: str, tenant_id: str) -> Optional[bytes]: ...


@runtime_checkable
class RealtimePublisher(Protocol):
    def publish(self, job_id: str, event_type: str, payload: Dict[str, Any]) -> None: ...

---
## Validation + Declarative Interface

In [ ]:
#| export
def validate_ai_interface_config(config: AIInterfaceConfig) -> List[str]:
    errors: List[str] = []
    if config.max_concurrency < 1:
        errors.append('max_concurrency must be >= 1')
    if config.default_poll_interval_ms < 100:
        errors.append('default_poll_interval_ms must be >= 100')
    if '{job_id}' not in config.status_endpoint_template:
        errors.append("status_endpoint_template must contain '{job_id}'")
    if '{job_id}' not in config.result_endpoint_template:
        errors.append("result_endpoint_template must contain '{job_id}'")
    if config.max_upload_mb < 1:
        errors.append('max_upload_mb must be >= 1')
    return errors


class Interface:
    '''Declarative AI interface wrapper for fh-saas.

    Phase 1 scope:
    - Captures fn/inputs/outputs metadata
    - Exposes route registration metadata
    - Builds polling contract for HTMX clients
    '''

    def __init__(self, fn: Callable[..., Any], inputs: List[Any], outputs: List[Any], *,
                 name: str = 'AI Interface', config: Optional[AIInterfaceConfig] = None):
        self.fn = fn
        self.inputs = inputs
        self.outputs = outputs
        self.name = name
        self.config = config or AIInterfaceConfig.from_env()

        errors = validate_ai_interface_config(self.config)
        if errors:
            raise ValueError('AI interface config errors: ' + '; '.join(errors))

    def describe(self) -> Dict[str, Any]:
        return {
            'name': self.name,
            'input_count': len(self.inputs),
            'output_count': len(self.outputs),
            'max_concurrency': self.config.max_concurrency,
            'storage_mode': self.config.storage_mode,
        }

    def build_polling_contract(self, job_id: str) -> Dict[str, Any]:
        return {
            'job_id': job_id,
            'status_endpoint': self.config.status_endpoint_template.format(job_id=job_id),
            'result_endpoint': self.config.result_endpoint_template.format(job_id=job_id),
            'poll_interval_ms': self.config.default_poll_interval_ms,
        }


def register_ai_routes(app, interface: Interface, *, submit_handler=None, status_handler=None, result_handler=None):
    '''Route orchestrator seam for AI interfaces.

    Caller wires concrete handlers from queue/runtime modules.
    '''
    if submit_handler is not None:
        app.post('/api/jobs')(submit_handler)
    if status_handler is not None:
        app.get('/api/jobs/{job_id}')(status_handler)
    if result_handler is not None:
        app.get('/api/jobs/{job_id}/result')(result_handler)

    logger.info('AI routes registered: /api/jobs, /api/jobs/{job_id}, /api/jobs/{job_id}/result')